# S1-04: 출력 제어 (프리필링 + 정지 시퀀스 + JSON 추출)
**Skilljar L12-L14: Controlling Model Output / Structured Data**

## 학습 목표
- 프리필링(Prefilling)으로 응답의 시작 방향을 유도한다
- 정지 시퀀스(Stop Sequences)로 응답 생성을 원하는 지점에서 중단한다
- 프리필 + 정지 시퀀스를 조합하여 순수 JSON 데이터를 추출한다
- 건축공학 비정형 데이터에서 구조화된 정보를 추출한다

In [ ]:
# 패키지 설치
%pip install anthropic python-dotenv

In [ ]:
# 환경변수 로드
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# 클라이언트 생성 및 헬퍼 함수
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"

def add_user_message(messages: list, text: str):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages: list, text: str):
    messages.append({"role": "assistant", "content": text})

def chat(messages: list, system: str = None, temperature: float = 1.0, stop_sequences: list = None) -> str:
    """Claude API 호출 (temperature + stop_sequences 포함)"""
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    return client.messages.create(**params).content[0].text

print("설정 완료")

## 1. 프리필링 (Prefilled Assistant Messages)

프리필링은 `assistant` 메시지의 시작 부분을 개발자가 미리 제공하여, Claude가 **그 지점부터 이어서 작성**하도록 유도하는 기법이다.

핵심 동작:
- Claude는 프리필된 텍스트를 **반복하지 않는다** (이미 말한 것으로 인식)
- 프리필의 방향을 유지하면서 이어서 생성한다
- 최종 출력 = 프리필 텍스트 + 생성된 텍스트 (개발자가 직접 결합)

In [ ]:
# 프리필링 없이 — Claude가 자유롭게 양쪽 입장을 다룸
messages_free = []
add_user_message(messages_free, "철골 구조와 RC 구조 중 어떤 것이 더 나은가?")
response_free = chat(messages_free)
print("=== 프리필링 없음 ===")
print(response_free)

In [ ]:
# 프리필링 적용 — "RC 구조가 더 나은데, 그 이유는" 으로 시작 강제
messages_prefill = []
add_user_message(messages_prefill, "철골 구조와 RC 구조 중 어떤 것이 더 나은가?")
add_assistant_message(messages_prefill, "RC 구조가 더 나은데, 그 이유는")  # 프리필

response_prefill = chat(messages_prefill)
print("=== 프리필링 적용 ===")
# 프리필 텍스트 + 생성된 텍스트를 직접 결합
print("RC 구조가 더 나은데, 그 이유는" + response_prefill)

In [ ]:
# 프리필링 활용: 응답 형식 지정
# "## 검토 결과" 로 시작하게 만들기

messages_format = []
add_user_message(messages_format, "500x500 RC 기둥의 축력비를 검토해줘. fck=24MPa, Pu=2500kN.")
add_assistant_message(messages_format, "## 검토 결과\n\n")  # 마크다운 헤더로 시작

response_format = chat(messages_format, temperature=0.0)
print("## 검토 결과\n")
print(response_format)

## 2. 정지 시퀀스 (Stop Sequences)

정지 시퀀스는 Claude가 응답 생성 중 특정 문자열을 만나면 **즉시 생성을 중단**하게 한다.

| 종료 조건 | `stop_reason` 값 |
|---|---|
| 자연 종료 | `"end_turn"` |
| 토큰 한도 | `"max_tokens"` |
| **정지 시퀀스** | `"stop_sequence"` |

In [ ]:
# 정지 시퀀스 기본 예시: 숫자 세기에서 "5"에서 중단
messages = []
add_user_message(messages, "1부터 10까지 쉼표로 구분하여 세어라.")
result = chat(messages, stop_sequences=["5"])
print(f"응답: {result}")
# 출력 예: "1, 2, 3, 4, " — 5는 포함되지 않음

In [ ]:
# stop_reason 확인 — 정지 시퀀스에 의해 중단되었는지 확인
messages = []
add_user_message(messages, "1부터 10까지 쉼표로 구분하여 세어라.")

response = client.messages.create(
    model=model,
    max_tokens=200,
    messages=messages,
    stop_sequences=["5"]
)

print(f"응답: {response.content[0].text}")
print(f"종료 사유: {response.stop_reason}")  # "stop_sequence"

In [ ]:
# 여러 정지 시퀀스를 동시에 지정할 수 있다
messages = []
add_user_message(messages, "건축구조 재료를 설명하라: 1) 콘크리트, 2) 철골, 3) 목재, 4) 조적")

result = chat(messages, stop_sequences=["3)"])  # 3) 앞에서 중단
print(f"응답 (3번 항목 전에 중단):\n{result}")

## 3. 프리필 + 정지 시퀀스 조합: 순수 JSON 추출

Claude에게 JSON을 요청하면 설명 텍스트가 함께 출력되는 문제가 있다.

**해결책:** 프리필 + 정지 시퀀스 콤보

```
1. 프리필: assistant에 ```json\n 삽입
2. Claude가 JSON 본문 생성
3. 정지 시퀀스: ``` 만나면 즉시 중단
4. 결과: 순수 JSON만 획득
```

In [ ]:
import json

# 문제 상황: JSON 요청 시 설명 텍스트가 함께 출력됨
messages_plain = []
add_user_message(messages_plain, "500x500 RC 기둥 정보를 JSON으로 출력해줘. fck=24MPa, 주근 8-D25.")

result_plain = chat(messages_plain)
print("=== 프리필 없이 JSON 요청 (설명 텍스트 포함 문제) ===")
print(result_plain[:500])  # 설명 텍스트 + JSON이 섞여 나옴

In [ ]:
# 해결: 프리필 + 정지 시퀀스로 순수 JSON만 추출

messages_json = []
add_user_message(messages_json,
    "500x500 RC 기둥 설계 결과를 JSON으로 출력하라. "
    "fck=24MPa, fy=400MPa, 주근 8-D25."
)
# 프리필: JSON 코드 블록의 시작 부분
add_assistant_message(messages_json, "```json\n")

# 정지 시퀀스: 코드 블록 종료 마커
raw_text = chat(messages_json, stop_sequences=["```"], temperature=0.0)

print("=== 추출된 순수 JSON ===")
print(raw_text)

# JSON 파싱
data = json.loads(raw_text.strip())
print("\n=== 파싱된 Python 딕셔너리 ===")
print(json.dumps(data, indent=2, ensure_ascii=False))

In [ ]:
# 다양한 형식에 같은 패턴 적용 가능

# CSV 형식 추출
messages_csv = []
add_user_message(messages_csv,
    "다음 3개 기둥의 철근비를 CSV로 정리하라: "
    "C1(500x500, 8-D25), C2(600x600, 12-D25), C3(400x400, 8-D22)"
)
add_assistant_message(messages_csv, "```csv\n")

csv_result = chat(messages_csv, stop_sequences=["```"], temperature=0.0)
print("=== 추출된 CSV ===")
print(csv_result)

## 4. 비정형 텍스트에서 구조 데이터 추출

실무에서 가장 유용한 패턴: 회의 메모, 현장 일지 등 비정형 텍스트에서 구조화된 JSON을 추출한다.

In [ ]:
import json

# 비정형 구조 검토 메모
unstructured_text = """
오늘 현장 미팅에서 3층 기둥 C3에 대해 논의함.
현재 단면 400x400인데 축력이 예상보다 크게 나옴.
김 소장이 500x500으로 변경 요청. 콘크리트는 30MPa로 상향.
철근은 8-D25에서 12-D29로 변경 필요할 듯.
다음 주 화요일까지 변경 도면 제출해야 함.
"""

messages = []
add_user_message(messages,
    f"다음 비정형 메모에서 구조 변경 정보를 JSON으로 추출하라:\n\n"
    f"{unstructured_text}\n\n"
    f"JSON 키: member_id, floor, original_section, revised_section, "
    f"original_rebar, revised_rebar, concrete_grade, deadline, requester"
)
add_assistant_message(messages, "```json\n")

raw = chat(messages, stop_sequences=["```"], temperature=0.0)
change_order = json.loads(raw.strip())

print("=== 추출된 설계 변경 정보 ===")
print(json.dumps(change_order, indent=2, ensure_ascii=False))

In [ ]:
# 추출된 데이터를 프로그래밍적으로 활용
print(f"변경 대상 부재: {change_order.get('member_id', 'N/A')}")
print(f"위치: {change_order.get('floor', 'N/A')}층")
print(f"단면 변경: {change_order.get('original_section', 'N/A')} -> {change_order.get('revised_section', 'N/A')}")
print(f"제출 기한: {change_order.get('deadline', 'N/A')}")

---
## 건축공학 실습 과제

### 과제: 구조 설계 결과 JSON 추출

아래 구조 설계 조건을 Claude에게 전달하고, **프리필 + 정지 시퀀스**를 사용하여 순수 JSON으로 설계 검토 결과를 추출하세요.

**설계 조건:**
- RC 기둥 3개 (C1, C2, C3)
- C1: 500x500, fck=24MPa, 8-D25, Pu=2500kN
- C2: 600x600, fck=27MPa, 12-D25, Pu=4000kN
- C3: 400x400, fck=24MPa, 8-D22, Pu=1500kN

**JSON 출력 형식:**
```json
[
  {
    "member_id": "C1",
    "section": "500x500",
    "axial_ratio": 0.xx,
    "reinforcement_ratio": 0.xx,
    "pass": true/false,
    "remark": "..."
  },
  ...
]
```

**요구사항:**
1. 프리필 + 정지 시퀀스 조합 사용
2. `temperature=0.0`
3. `json.loads()`로 파싱하여 Python 객체로 변환
4. 각 기둥의 판정 결과(pass/fail)를 출력

In [ ]:
import json

# TODO: 프리필 + 정지 시퀀스로 JSON 추출

messages = []
add_user_message(messages, """다음 RC 기둥 3개의 설계 적정성을 검토하고 JSON 배열로 출력하라.

기둥 목록:
- C1: 500x500, fck=24MPa, fy=400MPa, 주근 8-D25, Pu=2500kN
- C2: 600x600, fck=27MPa, fy=400MPa, 주근 12-D25, Pu=4000kN
- C3: 400x400, fck=24MPa, fy=400MPa, 주근 8-D22, Pu=1500kN

각 기둥에 대해 다음 키를 포함하라:
- member_id: 기둥 ID
- section: 단면 크기 (mm)
- axial_ratio: 축력비 (소수점 3자리)
- reinforcement_ratio: 철근비 (소수점 4자리)
- pass: 적합 여부 (true/false)
- remark: 비고""")

# 프리필: JSON 배열 시작
add_assistant_message(messages, "```json\n")

# API 호출 (정지 시퀀스로 JSON만 추출)
raw_json = chat(messages, stop_sequences=["```"], temperature=0.0)

# JSON 파싱
columns = json.loads(raw_json.strip())

print("=== 구조 설계 검토 결과 (JSON) ===")
print(json.dumps(columns, indent=2, ensure_ascii=False))

# 판정 결과 요약
print("\n=== 판정 요약 ===")
for col in columns:
    status = "PASS" if col.get("pass") else "FAIL"
    print(f"  {col['member_id']} ({col['section']}): {status} - {col.get('remark', '')}")